# Putting the patterns together: a mini-Xception

Residual connections, batch normalization, and separable convolutions in one architecture — the model the rest of the vision chapters build on.

**Runs on:** GPU recommended — about 15 minutes on CPU &nbsp;·&nbsp; **Slides:** [Chapter 9 — ConvNet Architecture Patterns](../../../course-web-slides/ch09/index.html) &nbsp;·&nbsp; **Section:** 05 — Putting it together

---

## The three patterns, assembled

In [ ]:
import keras
from keras import layers

def mini_xception(input_shape=(180, 180, 3), num_classes=1):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)

    # A regular convolution first: three channels are not worth separating.
    x = layers.Conv2D(32, 5, use_bias=False)(x)

    for size in [32, 64, 128, 256, 512]:
        residual = x

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        residual = layers.Conv2D(size, 1, strides=2, padding="same",
                                 use_bias=False)(residual)
        x = layers.add([x, residual])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    activation = "sigmoid" if num_classes == 1 else "softmax"
    outputs = layers.Dense(num_classes, activation=activation)(x)
    return keras.Model(inputs, outputs)

model = mini_xception()
print(f"{model.count_params():,} parameters, {len(model.layers)} layers")

Five blocks, each: **normalize, activate, separable-convolve** — twice — then pool, then add the projected shortcut.

Note the ordering inside the block: normalization and activation come **before** the convolution, not after. This is the *pre-activation* arrangement, and it keeps the residual path completely clean — nothing but additions from input to output.

## Filter counts grow as the maps shrink

In [ ]:
for l in model.layers:
    if isinstance(l, (layers.SeparableConv2D, layers.MaxPooling2D)):
        print(f"{l.__class__.__name__:18s} {str(l.output.shape):24s}")

180 → 88 → 44 → 22 → 11 → 6, while filters go 32 → 512. **The same trade as chapter 8**, applied more aggressively.

## Training it on cats and dogs

In [ ]:
import pathlib
from keras.utils import image_dataset_from_directory

new_base_dir = pathlib.Path("cats_vs_dogs_small")
train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=(180, 180), batch_size=32)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=(180, 180), batch_size=32)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=(180, 180), batch_size=32)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
])

inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
outputs = mini_xception()(x)
model = keras.Model(inputs, outputs)

model.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])
callbacks = [keras.callbacks.ModelCheckpoint("mini_xception.keras",
                                             save_best_only=True,
                                             monitor="val_loss")]
history = model.fit(train_dataset, epochs=60,
                    validation_data=validation_dataset,
                    callbacks=callbacks, verbose=2)

## Against chapter 8's from-scratch model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

h = history.history
plt.figure(figsize=(7, 4.4))
plt.plot(h["accuracy"], lw=1, label="training")
plt.plot(h["val_accuracy"], lw=1.7, label="validation")
plt.axhline(0.83, ls="--", c="k", lw=1,
            label="chapter 8, augmented, from scratch")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.title("Mini-Xception on the same 2,000 images")
plt.show()

best = keras.models.load_model("mini_xception.keras")
print(f"test accuracy: {best.evaluate(test_dataset, verbose=0)[1]:.3f}")

Expected output:

```
test accuracy: ~0.88 to 0.90
```

About 90%, against 83% for chapter 8's plain stack on identical data. **Architecture is worth roughly seven points here** — and still not the 97% a pretrained backbone gives, which remains the chapter 8 lesson.

## One block at a time: what each pattern contributes

In [ ]:
import itertools

def variant(residual=True, bn=True, separable=True, blocks=(32, 64, 128)):
    keras.utils.set_random_seed(0)
    Conv = layers.SeparableConv2D if separable else layers.Conv2D
    i = keras.Input(shape=(180, 180, 3))
    x = layers.Rescaling(1./255)(i)
    x = layers.Conv2D(32, 5, use_bias=False)(x)
    for size in blocks:
        r = x
        if bn: x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = Conv(size, 3, padding="same", use_bias=False)(x)
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
        if residual:
            r = layers.Conv2D(size, 1, strides=2, padding="same",
                              use_bias=False)(r)
            x = layers.add([x, r])
    x = layers.GlobalAveragePooling2D()(x)
    o = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(i, o)
    m.compile(loss="binary_crossentropy", optimizer="rmsprop",
              metrics=["accuracy"])
    return m

print("Run each of these for ~20 epochs and compare. Expect the ordering to")
print("hold even if the absolute numbers differ on your hardware:")
for name, kw in [("all three", {}),
                 ("no residual", {"residual": False}),
                 ("no batchnorm", {"bn": False}),
                 ("regular convolutions", {"separable": False})]:
    m = variant(**kw)
    print(f"  {name:22s} {m.count_params():>9,} parameters")

Running the full comparison takes about an hour; the parameter counts alone are informative, and the exercise is worth doing once on your own hardware. **Ablation is how you find out which of your ideas were doing the work** — chapter 18 formalises it.

---

## What to take away

- Mini-Xception is the three patterns composed: pre-activation, separable convolutions, projected residuals.
- Normalization and activation go **before** the convolution, keeping the residual path clean.
- Architecture is worth about seven points here — real, and smaller than pretraining.
- Ablate one pattern at a time to find out which of them was doing the work.